# Gastro-AI: Entrenamiento con Pérdida Enmascarada (Masked Loss)
Este cuaderno implementa **Pérdida Enmascarada (Masked BCEWithLogitsLoss)** para corregir el desajuste de etiquetas en **EGID** y eliminar los falsos positivos ficticios.

In [ ]:
# ==============================================================================
# CELDA 1: MONTAJE DE GOOGLE DRIVE Y CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
import os
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
from PIL import Image
from sklearn.metrics import roc_auc_score, f1_score
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('✓ Google Drive montado correctamente.')

BASE_DIR = '/content/drive/MyDrive/GastroAI'
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
SPLITS_DIR = os.path.join(BASE_DIR, 'splits')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')

for folder in [DATASET_DIR, SPLITS_DIR, REPORTS_DIR]:
    os.makedirs(folder, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Dispositivo activo: {device}')
if device.type == 'cuda':
    print(f'  • GPU: {torch.cuda.get_device_name(0)}')

# Semillas fijas para reproducibilidad (RM-07)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)
print('✓ Semillas fijas configuradas (seed=42) para reproducibilidad.')


In [ ]:
# ==============================================================================
# CELDA 2: CARGA DE DATOS Y CONSTRUCCIÓN DE LA MÁSCARA DE ETIQUETAS
# ==============================================================================
splits_csv_path = os.path.join(SPLITS_DIR, 'patient_splits_5fold.csv')
annotations_csv_path = os.path.join(DATASET_DIR, 'annotations.csv')

if os.path.exists(splits_csv_path):
    df_dataset = pd.read_csv(splits_csv_path)
elif os.path.exists(annotations_csv_path):
    df_dataset = pd.read_csv(annotations_csv_path)
else:
    raise FileNotFoundError('❌ No se hallaron archivos de annotations o splits en Drive.')

target_cols = ['normal_mucosa', 'erythematous_gastritis', 'gastric_atrophy', 'h_pylori_signs']

# Definición de la máscara de presencia de etiqueta real (1 = Anotado, 0 = Ignorar en pérdida)
def generate_label_masks(df):
    masks = []
    for _, row in df.iterrows():
        path_str = str(row['image_path']).lower()
        ds = str(row.get('dataset', '')).upper()
        
        # Por defecto, todas las clases están anotadas (1, 1, 1, 1)
        m = [1.0, 1.0, 1.0, 1.0]
        
        # En EGID, enmascarar clases no evaluadas según la carpeta de origen
        if 'egid' in path_str or ds == 'EGID':
            if '1.h_pylori' in path_str or 'h_pylori_infection_status' in path_str:
                # Esta carpeta solo evalúa H. pylori / Gastritis / Normal. Atrofia NO está anotada.
                m[2] = 0.0  # Mask out gastric_atrophy
            elif '2.presence_of_atrophy' in path_str or 'atrophy' in path_str:
                # Esta carpeta solo evalúa Atrofia. Gastritis y H. pylori NO están anotadas.
                m[1] = 0.0  # Mask out erythematous_gastritis
                m[3] = 0.0  # Mask out h_pylori_signs
                
        masks.append(m)
    return np.array(masks, dtype=np.float32)

masks_array = generate_label_masks(df_dataset)
for i, col in enumerate(target_cols):
    df_dataset[f'mask_{col}'] = masks_array[:, i]
    print(f'• {col:<22}: {int(masks_array[:, i].sum())}/{len(df_dataset)} muestras supervisadas.')


In [ ]:
# ==============================================================================
# CELDA 3: DATASET Y DATALOADERS ENMASCARADOS EN PYTORCH
# ==============================================================================
class MaskedGastroAIDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.image_paths = self.df['image_path'].values
        self.labels = self.df[target_cols].values.astype(np.float32)
        self.masks = self.df[[f'mask_{c}' for c in target_cols]].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        mask = self.masks[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (240, 240), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label), torch.tensor(mask)

train_transforms = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_df = df_dataset[df_dataset['fold'] != 0]
val_df = df_dataset[df_dataset['fold'] == 0]

train_dataset = MaskedGastroAIDataset(train_df, transform=train_transforms)
val_dataset = MaskedGastroAIDataset(val_df, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'✓ DataLoaders listos con Masking: Train={len(train_dataset)} | Val={len(val_dataset)}')


In [ ]:
# ==============================================================================
# CELDA 4: MODELO, PÉRDIDA ENMASCARADA (MASKED BCE) Y LOOP DE ENTRENAMIENTO
# ==============================================================================
class GastroAI_EfficientNetB1(nn.Module):
    def __init__(self, num_classes=4, pretrained=True):
        super(GastroAI_EfficientNetB1, self).__init__()
        weights = EfficientNet_B1_Weights.DEFAULT if pretrained else None
        self.backbone = efficientnet_b1(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

model = GastroAI_EfficientNetB1(num_classes=4, pretrained=True).to(device)

# Función de pérdida enmascarada (Masked BCEWithLogitsLoss)
class MaskedBCEWithLogitsLoss(nn.Module):
    def __init__(self):
        super(MaskedBCEWithLogitsLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        
    def forward(self, logits, targets, masks):
        loss_elementwise = self.bce(logits, targets)
        masked_loss = loss_elementwise * masks
        return masked_loss.sum() / torch.clamp(masks.sum(), min=1.0)

criterion = MaskedBCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

def train_masked_gastroai(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=10):
    history = {'train_loss': [], 'val_loss': [], 'val_auc_micro': [], 'val_f1_micro': []}
    best_val_loss = float('inf')
    checkpoint_path = os.path.join(BASE_DIR, 'best_gastroai_masked_loss.pth')
    
    print(f'\n🚀 Iniciando entrenamiento con Pérdida Enmascarada por {num_epochs} épocas...')
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        model.train()
        running_train_loss = 0.0
        
        for images, labels, masks in train_loader:
            images, labels, masks = images.to(device), labels.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels, masks)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * images.size(0)
            
        train_loss = running_train_loss / len(train_loader.dataset)
        
        # Validación
        model.eval()
        running_val_loss = 0.0
        all_targets, all_probs, all_masks = [], [], []
        
        with torch.no_grad():
            for images, labels, masks in val_loader:
                images, labels, masks = images.to(device), labels.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels, masks)
                running_val_loss += loss.item() * images.size(0)
                
                probs = torch.sigmoid(outputs)
                all_targets.append(labels.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
                all_masks.append(masks.cpu().numpy())
                
        val_loss = running_val_loss / len(val_loader.dataset)
        scheduler.step(val_loss)
        
        all_targets = np.vstack(all_targets)
        all_probs = np.vstack(all_probs)
        all_masks = np.vstack(all_masks)
        
        # Evaluar métricas solo sobre elementos supervisados (mask == 1)
        valid_indices = (all_masks == 1.0)
        all_preds = (all_probs >= 0.5).astype(int)
        
        try:
            val_auc = roc_auc_score(all_targets[valid_indices], all_probs[valid_indices])
        except Exception:
            val_auc = 0.5
            
        val_f1 = f1_score(all_targets[valid_indices], all_preds[valid_indices], zero_division=0)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc_micro'].append(val_auc)
        history['val_f1_micro'].append(val_f1)
        
        epoch_time = time.time() - epoch_start
        print(f'Época [{epoch+1:02d}/{num_epochs:02d}] ({epoch_time:.1f}s) | '
              f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
              f'Val ROC-AUC (Masked): {val_auc:.4f} | Val F1 (Masked): {val_f1:.4f}')
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
            print(f'  ✓ Modelo con Pérdida Enmascarada guardado en: {checkpoint_path}')
            
    total_time = (time.time() - start_time) / 60
    print(f'\n✓ Entrenamiento finalizado en {total_time:.2f} minutos.')
    return history

history = train_masked_gastroai(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=10)


In [ ]:
# ==============================================================================
# CELDA 5: GRÁFICAS COMPARATIVAS DE PÉRDIDA Y RENDIMIENTO
# ==============================================================================
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Pérdida Entrenamiento (Masked)', color='#1f77b4', linewidth=2)
plt.plot(history['val_loss'], label='Pérdida Validación (Masked)', color='#ff7f0e', linewidth=2)
plt.title('Pérdida Enmascarada (Masked BCEWithLogitsLoss)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(history['val_auc_micro'], label='ROC-AUC Supervisado', color='#2ca02c', linewidth=2)
plt.plot(history['val_f1_micro'], label='F1-Score Supervisado', color='#9467bd', linewidth=2)
plt.title('Métricas Clínicas Reales (Sin Penalización Ficticia)')
plt.xlabel('Época')
plt.ylabel('Puntaje')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
